In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


ROOT = Path(".").resolve()       # jesteś w Results
print(ROOT)
OUT = ROOT / "figures_feature_selection"

TRANSITIONS = ["AC", "AB", "BCb"]
FEATURE_SETS = [30, 20, 12]

CALIBRATION = "uncalibrated"

FORMATS = ["pdf", "png"]
DPI = 160

OBSERVABLE = {
    "AC": r"$K_0$",
    "AB": r"$\Delta$",
    "BCb": r"$\Delta$",
}

# Kolory modeli będą przypisywane automatycznie.
COLORS = [
    "#1f77b4",
    "#d62728",
    "#2ca02c",
    "#ff7f0e",
    "#9467bd",
    "#8c564b",
    "#17becf",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
]

MARKERS = [
    "o", "s", "^", "D", "v",
    "P", "X", "*", "<", ">"
]

/home/mariuszoslaw/uni/masters/Results


In [2]:
Y_LABELS = {
    "AC": r"$\langle \Pr(d_j(K_0)\in C)\rangle$",
    "AB": r"$\langle \Pr(d_j(\Delta)\in A)\rangle$",
    "BCb": r"$\langle \Pr(d_j(\Delta)\in C_b)\rangle$",
}

In [3]:
MODEL_NAMES_PL = {
    # -------------------------
    # Supervised
    # -------------------------
    "Logistic Regression": "Regresja logistyczna",
    "Decision Tree": "Drzewo decyzyjne",
    "Random Forest": "Las losowy",
    "Gradient Boosted Trees": "Wzmocnienie gradientowe",
    "kNN": "kNN",
    "SVM (RBF)": "SVM (RBF)",
    "Neural Network": "Sieć neuronowa",

    # -------------------------
    # Unsupervised
    # -------------------------
    "KMeans": "K-średnich",
    "Agglomerative": "Grupowanie aglomeracyjne",
    "Spectral": "Grupowanie spektralne",
    "GaussianMixture": "Mieszanina Gaussowska",
    "MeanShift": "Przesunięcie średniej",
    "Birch": "BIRCH",
    "BayesianGMM": "Bayesowska MG",
    "DBSCAN": "DBSCAN",
}

In [4]:
def load_transition(root, method, transition):
    """
    Wczytuje curve_mean i summary dla danego:
        method = SUPERVISED / UNSUPERVISED
        transition = AC / AB / BCb
    """

    folder = root / method / transition

    if method == "SUPERVISED":
        prefix = transition
    elif method == "UNSUPERVISED":
        prefix = f"{transition}_UNS"
    else:
        raise ValueError(f"Nieznana metoda: {method}")

    mean_path = folder / f"{prefix}_curve_mean.csv"
    summary_path = folder / f"{prefix}_summary.csv"

    mean_df = None
    summary_df = None

    if mean_path.exists():
        mean_df = pd.read_csv(mean_path)
    else:
        print(f"[brak] {mean_path}")

    if summary_path.exists():
        summary_df = pd.read_csv(summary_path)
    else:
        print(f"[brak] {summary_path}")

    return mean_df, summary_df

In [5]:
def save_figure(fig, name):
    OUT.mkdir(parents=True, exist_ok=True)

    for ext in FORMATS:
        path = OUT / f"{name}.{ext}"

        fig.savefig(
            path,
            dpi=DPI,
            bbox_inches="tight",
        )

        print(f"  zapisano: {path}")

    plt.close(fig)

In [6]:
def get_model_colors(dataframes):
    models = []

    for df in dataframes:
        if df is None or "model" not in df.columns:
            continue

        for model in pd.unique(df["model"]):
            if model not in models:
                models.append(model)

    return {
        model: COLORS[i % len(COLORS)]
        for i, model in enumerate(models)
    }


def get_model_markers(dataframes):
    models = []

    for df in dataframes:
        if df is None or "model" not in df.columns:
            continue

        for model in pd.unique(df["model"]):
            if model not in models:
                models.append(model)

    return {
        model: MARKERS[i % len(MARKERS)]
        for i, model in enumerate(models)
    }

In [7]:
def filter_supervised_models(df):
    df = df.copy()

    # Zachowujemy dokładnie jeden wariant Logistic Regression
    # oraz dokładnie jeden wariant kNN.

    keep = (
        (df["model"] == "Logistic Regression")
        |
        (df["model"] == "kNN")
        |
        (
            ~df["model"].str.startswith("Logistic Regression")
            &
            ~df["model"].str.startswith("kNN")
        )
    )

    return df[keep].copy()

In [8]:
def plot_panel(
    ax,
    mean_df,
    summary_df,
    transition,
    feature_set,
    model_colors,
    model_markers,
    method,
    linestyle="-",
    prefix=None,
):
    # =====================================================
    # KOLUMNA OSI X
    # =====================================================

    if transition == "AC":
        x_col = "K0"
    else:
        x_col = "Delta"

    # =====================================================
    # 1. FILTROWANIE CURVE_MEAN
    # =====================================================

    df = mean_df[
        mean_df["feature_set"] == feature_set
    ].copy()

    if method == "SUPERVISED":

        # tylko uncalibrated
        df = df[
            df["calibration"] == "uncalibrated"
        ].copy()

        # wybrane modele
        df = filter_supervised_models(df)

    elif method == "UNSUPERVISED":

        # dla każdego modelu najmniejszy stride
        min_stride = (
            df.groupby("model")["stride"]
            .transform("min")
        )

        df = df[
            df["stride"] == min_stride
        ].copy()

    else:
        raise ValueError(
            f"Nieznana metoda: {method}"
        )

    # =====================================================
    # BRAK DANYCH
    # =====================================================

    if df.empty:
        ax.text(
            0.5,
            0.5,
            "Brak danych",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return

    # =====================================================
    # 2. SUMMARY
    # =====================================================

    if transition == "AC":
        crit_col = "K0_crit"

    else:
        crit_col = "Delta_crit"

    if summary_df is not None:

        summ = summary_df[
            summary_df["feature_set"] == feature_set
        ].copy()

        if method == "SUPERVISED":

            summ = summ[
                summ["calibration"] == "uncalibrated"
            ].copy()

            summ = filter_supervised_models(summ)

        elif method == "UNSUPERVISED":

            min_stride = (
                summ.groupby("model")["stride"]
                .transform("min")
            )

            summ = summ[
                summ["stride"] == min_stride
            ].copy()

    else:
        summ = pd.DataFrame()

    # =====================================================
    # 3. CZY ODWRACAMY PRAWDOPODOBIEŃSTWO?
    # =====================================================
    #
    # AC:
    #     zawsze 1 - p
    #
    # Supervised + 20 cech:
    #     również 1 - p
    #
    # =====================================================

    invert_probability = (
        transition == "AC"
        or (
            method == "SUPERVISED" and transition == "AB"
        )
    )

    # =====================================================
    # 4. RYSOWANIE KRZYWYCH
    # =====================================================

    for model in pd.unique(df["model"]):

        group = (
            df[df["model"] == model]
            .sort_values(x_col)
            .copy()
        )

        # ---------------------------------------------
        # prawdopodobieństwo
        # ---------------------------------------------

        y = group["mean"].to_numpy()

        if invert_probability:
            y = 1.0 - y

        # ---------------------------------------------
        # błąd
        #
        # dla p -> 1-p błąd bezwzględny pozostaje taki sam
        # ---------------------------------------------

        yerr = group["mean_err"].to_numpy()

        color = model_colors[model]
        marker = model_markers[model]

        label = MODEL_NAMES_PL.get(
            model,
            model,
        )

        if prefix is not None:
            label = f"{prefix} — {label}"

        ax.errorbar(
            group[x_col],
            y,
            yerr=yerr,
            fmt=marker,
            linestyle=linestyle,
            markersize=3,
            capsize=2,
            linewidth=1.1,
            color=color,
            label=label,
        )

    # =====================================================
    # 5. WARTOŚCI KRYTYCZNE
    # =====================================================

    if (
        not summ.empty
        and crit_col in summ.columns
    ):

        crit_values = (
            pd.to_numeric(
                summ[crit_col],
                errors="coerce",
            )
            .dropna()
        )

        if not crit_values.empty:

            lo = crit_values.min()
            hi = crit_values.max()

            # zakres wartości krytycznych
            ax.axvspan(
                lo,
                hi,
                color="gray",
                alpha=0.15,
                zorder=0,
            )

            # średnia wartość krytyczna
            mean_crit = crit_values.mean()

            ax.axvline(
                mean_crit,
                color="black",
                linestyle=":",
                linewidth=1.0,
                zorder=1,
            )
            ax.text(
                mean_crit,
                1.02,
                f"{mean_crit:.4g}",
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=7,
                clip_on=False,
            )

    # =====================================================
    # 6. OŚ X — AB: TYLKO CO TRZECIA ETYKIETA
    # =====================================================

    if transition in ["AB"]:
      
        plt.setp(
            ax.get_xticklabels(),
            rotation=45,
            ha="right",
        )

    # =====================================================
    # 7. WYGLĄD
    # =====================================================

    ax.grid(
        alpha=0.3,
        linewidth=0.6,
    )

    ax.set_xlabel(
        r"$K_0$"
        if transition == "AC"
        else r"$\Delta$"
    )

    ax.set_ylabel(
        Y_LABELS[transition]
    )

In [9]:
def make_feature_selection_figure(
    root,
    group,
    filename
):
    """
    Generuje macierz 3 x 3 wykresów:

                    AC              AB              BCb
        30 cech     [ ]             [ ]             [ ]
        20 cech     [ ]             [ ]             [ ]
        12 cech     [ ]             [ ]             [ ]

    Parameters
    ----------
    root : Path
        Katalog Results.

    group : str
        "SUPERVISED"
        "UNSUPERVISED"
        "COMBINED"

    filename : str
        Nazwa wynikowego pliku bez rozszerzenia.

    title : str
        Tytuł całej figury.
    """

    root = Path(root)

    # =====================================================
    # 1. WCZYTANIE DANYCH
    # =====================================================

    datasets = {}

    for transition in TRANSITIONS:

        # ---------------------------------------------
        # SUPERVISED
        #
        # np.
        # Results/SUPERVISED/AC/AC_summary.csv
        # Results/SUPERVISED/AC/AC_curve_mean.csv
        # ---------------------------------------------

        sup_mean, sup_summary = load_transition(
            root,
            "SUPERVISED",
            transition,
        )
        # ---------------------------------------------
        # UNSUPERVISED
        #
        # np.
        # Results/UNSUPERVISED/AC/AC_UNS_summary.csv
        # Results/UNSUPERVISED/AC/AC_UNS_curve_mean.csv
        # ---------------------------------------------

        uns_mean, uns_summary = load_transition(
            root,
            "UNSUPERVISED",
            transition,
        )

        datasets[transition] = {
            "supervised_mean": sup_mean,
            "supervised_summary": sup_summary,
            "unsupervised_mean": uns_mean,
            "unsupervised_summary": uns_summary,
        }

    # =====================================================
    # 2. ZBIERAMY WSZYSTKIE MODELE
    # =====================================================

    all_mean = []

    for transition in TRANSITIONS:

        data = datasets[transition]

        if group in ["SUPERVISED", "COMBINED"]:

            if data["supervised_mean"] is not None:
                all_mean.append(
                    data["supervised_mean"]
                )

        if group in ["UNSUPERVISED", "COMBINED"]:

            if data["unsupervised_mean"] is not None:
                all_mean.append(
                    data["unsupervised_mean"]
                )

    if not all_mean:
        print("Brak danych do narysowania.")
        return

    # =====================================================
    # 3. KOLORY I MARKERY MODELI
    # =====================================================

    filtered_all_mean = []

    for df in all_mean:
    
        if "calibration" in df.columns:
            df = df[
                df["calibration"] == "uncalibrated"
            ].copy()
    
            df = filter_supervised_models(df)
    
        filtered_all_mean.append(df)
    
    model_colors = get_model_colors(filtered_all_mean)
    model_markers = get_model_markers(filtered_all_mean)

    # =====================================================
    # 4. FIGURA 3 x 3
    # =====================================================
    
    fig, axes = plt.subplots(
        nrows=len(TRANSITIONS),      # AC, AB, BCb
        ncols=len(FEATURE_SETS),     # 30, 20, 12
        figsize=(15, 12),
        sharey=True,
    )
    
    
    # =====================================================
    # 5. RYSOWANIE PANELI
    # =====================================================
    
    for row, transition in enumerate(TRANSITIONS):
    
        data = datasets[transition]
    
        sup_mean = data["supervised_mean"]
        sup_summary = data["supervised_summary"]
    
        uns_mean = data["unsupervised_mean"]
        uns_summary = data["unsupervised_summary"]
    
        for col, feature_set in enumerate(FEATURE_SETS):
    
            ax = axes[row, col]
    
            # =============================================
            # SUPERVISED
            # =============================================
    
            if group in ["SUPERVISED", "COMBINED"]:
    
                if sup_mean is not None:
    
                    plot_panel(
                        ax=ax,
                        mean_df=sup_mean,
                        summary_df=sup_summary,
                        transition=transition,
                        feature_set=feature_set,
                        model_colors=model_colors,
                        model_markers=model_markers,
                        method="SUPERVISED",
                        linestyle="-",
                        prefix=(
                            "S"
                            if group == "COMBINED"
                            else None
                        ),
                    )
    
            # =============================================
            # UNSUPERVISED
            # =============================================
    
            if group in ["UNSUPERVISED", "COMBINED"]:
    
                if uns_mean is not None:
    
                    plot_panel(
                        ax=ax,
                        mean_df=uns_mean,
                        summary_df=uns_summary,
                        transition=transition,
                        feature_set=feature_set,
                        model_colors=model_colors,
                        model_markers=model_markers,
                        method="UNSUPERVISED",
                        linestyle="--",
                        prefix=(
                            "U"
                            if group == "COMBINED"
                            else None
                        ),
                    )
    
    
    # =====================================================
    # 6. NAGŁÓWKI KOLUMN
    # =====================================================
    
    for col, feature_set in enumerate(FEATURE_SETS):
    
        axes[0, col].set_title(
            f"{feature_set} cech",
            fontsize=13,
            fontweight="bold",
            pad=12,
        )
    
    
    # =====================================================
    # 7. OPISY WIERSZY
    # =====================================================
    
    for row, transition in enumerate(TRANSITIONS):
    
        axes[row, 0].annotate(
            transition,
            xy=(0, 0.5),
            xytext=(-0.15, 0.5),
            xycoords="axes fraction",
            textcoords="axes fraction",
            rotation=90,
            va="center",
            ha="center",
            fontsize=13,
            fontweight="bold",
        )

    # =====================================================
    # 8. WSPÓLNA LEGENDA
    # =====================================================

    handles = []
    labels = []

    # ---------------------------------------------
    # legenda modeli
    # ---------------------------------------------

    for model in model_colors:

        handles.append(
            Line2D(
                [],
                [],
                color=model_colors[model],
                marker=model_markers[model],
                linestyle="-",
                markersize=5,
                linewidth=1.2,
            )
        )
        labels.append(MODEL_NAMES_PL.get(model, model))

    # ---------------------------------------------
    # supervised / unsupervised
    # ---------------------------------------------

    if group == "COMBINED":

        handles.extend(
            [
                Line2D(
                    [],
                    [],
                    color="black",
                    linestyle="-",
                    linewidth=1.5,
                ),
                Line2D(
                    [],
                    [],
                    color="black",
                    linestyle="--",
                    linewidth=1.5,
                ),
            ]
        )

        labels.extend(
            [
                "modele nadzorowane",
                "modele nienadzorowane",
            ]
        )

    # ---------------------------------------------
    # średnia wartość krytyczna
    # ---------------------------------------------

    handles.append(
        Line2D(
            [],
            [],
            color="black",
            linestyle=":",
            linewidth=1.0,
        )
    )

    labels.append(
        "średnia wartość krytyczna"
    )

    # ---------------------------------------------
    # zakres wartości krytycznych
    # ---------------------------------------------

    handles.append(
        plt.Rectangle(
            (0, 0),
            1,
            1,
            facecolor="gray",
            alpha=0.15,
            edgecolor="none",
        )
    )

    labels.append(
        "zakres wartości krytycznych modeli"
    )

    # ---------------------------------------------
    # legenda
    # ---------------------------------------------

    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.005),
        ncol=4,
        fontsize=8,
        frameon=False,
    )

    # =====================================================
    # 9. TYTUŁ
    # =====================================================

    # fig.suptitle(
    #     title,
    #     fontsize=16,
    #     y=0.995,
    # )

    # =====================================================
    # 10. UKŁAD
    # =====================================================

    plt.tight_layout(
        rect=[
            0.05,   # lewy
            0.09,   # dół — miejsce na legendę
            1.00,   # prawy
            0.96,   # góra — miejsce na tytuł
        ]
    )

    # =====================================================
    # 11. ZAPIS
    # =====================================================

    save_figure(
        fig,
        filename,
    )

In [10]:
make_feature_selection_figure(
    ROOT,
    "SUPERVISED",
    "feature_selection_supervised",
)

  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_feature_selection/feature_selection_supervised.pdf
  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_feature_selection/feature_selection_supervised.png


In [11]:
make_feature_selection_figure(
    ROOT,
    "UNSUPERVISED",
    "feature_selection_unsupervised",
)

  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_feature_selection/feature_selection_unsupervised.pdf
  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_feature_selection/feature_selection_unsupervised.png


In [12]:
make_feature_selection_figure(
    ROOT,
    "COMBINED",
    "feature_selection_combined",
)

  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_feature_selection/feature_selection_combined.pdf
  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_feature_selection/feature_selection_combined.png
